# -*- coding: utf-8 -*-
"""
GNINA_Virtual_Screening_KOR_Final_v2.ipynb
Author: Babak Mamnoon | Refined for ChEMBL-style CSVs
"""
# ============================================================
# SECTION 1 — INSTALL PACKAGES & DOWNLOAD GNINA
# ============================================================

In [ ]:
!pip -q install py3Dmol rdkit pandas numpy tqdm
!apt-get -qq install openbabel
!wget -q https://github.com/gnina/gnina/releases/download/v1.0.3/gnina
!chmod +x gnina

import os, re, glob, subprocess
import pandas as pd
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

# ============================================================
# SECTION 2 — SETUP WORKING DIRECTORY
# ============================================================

In [ ]:
WORKDIR = "/content/KOR_HTVS"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

# ============================================================
# SECTION 3 — RECEPTOR PREPARATION (4DJH)
# ============================================================

In [ ]:
!wget -q https://files.rcsb.org/download/4DJH.pdb

with open("4DJH.pdb", "r") as f:
    lines = f.readlines()

with open("receptor_clean.pdb", "w") as f:
    for line in lines:
        if line.startswith(("ATOM", "HETATM")):
            chain = line[21].strip()
            res_name = line[17:20].strip()
            if chain == "A":
                # Filter out salts and water, keep ligand JDC
                if res_name not in ["HOH", "EDO", "SO4", "GOL", "CIT"]:
                    f.write(line)
        elif line.startswith("TER"):
            f.write(line)

# Extract reference for the search box
!grep "JDC" receptor_clean.pdb > reference_ligand.pdb
# Add hydrogens to protein
!obabel receptor_clean.pdb -O receptor_prepared.pdb -h

# ============================================================
# SECTION 4 — LIGAND LIBRARY CLEANING & PREPARATION
# ============================================================

In [ ]:
# YOUR GITHUB URL
GITHUB_CSV_URL = "https://raw.githubusercontent.com/Babakmamnoon/High-Throughput-Virtual-Screening-of-Human-Kappa-Opioid-Receptor/refs/heads/main/ligands.csv"
!wget -q -O ligands_raw.csv $GITHUB_CSV_URL

# Load and handle redundancy
df_raw = pd.read_csv("ligands_raw.csv")

# 1. Identify columns (Supports both ChEMBL and generic formats)
id_col = 'molecule_chembl_id' if 'molecule_chembl_id' in df_raw.columns else df_raw.columns[0]
smiles_col = 'canonical_smiles' if 'canonical_smiles' in df_raw.columns else df_raw.columns[1]

# 2. Clean: Remove missing SMILES, remove duplicates, keep only ID and SMILES
df = df_raw.dropna(subset=[smiles_col]).copy()
df = df.drop_duplicates(subset=[smiles_col])
df = df[[id_col, smiles_col]]
df.columns = ['compound_id', 'smiles']

print(f"Library processed: {len(df_raw)} raw rows -> {len(df)} unique, valid ligands.")

# 3. Generate 3D Conformations
os.makedirs("ligands_sdf", exist_ok=True)
valid_ligand_paths = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="3D Conformation Gen"):
    try:
        mol = Chem.MolFromSmiles(str(row['smiles']), sanitize=True)
        if mol is None: continue
        
        mol = Chem.AddHs(mol)
        # Using ETKDGv3 (Experimental-Torsion-Knowledge Distance Geometry)
        if AllChem.EmbedMolecule(mol, AllChem.ETKDGv3()) == 0:
            AllChem.MMFFOptimizeMolecule(mol)
            # Add the ID as a property inside the SDF
            mol.SetProp("_Name", str(row['compound_id']))
            
            out_file = f"ligands_sdf/{row['compound_id']}.sdf"
            writer = Chem.SDWriter(out_file)
            writer.write(mol)
            writer.close()
            valid_ligand_paths.append(out_file)
    except:
        continue

# ============================================================
# SECTION 5 — HIGH-THROUGHPUT VIRTUAL SCREENING (GNINA)
# ============================================================

In [ ]:
os.makedirs("docking_results", exist_ok=True)

for ligand_path in tqdm(valid_ligand_paths, desc="Docking"):
    l_name = os.path.basename(ligand_path).replace(".sdf", "")
    output_file = f"docking_results/{l_name}_docked.sdf"
    
    # GNINA execution
    # --cnn_scoring refined: High accuracy rescoring
    # --num_modes 1: We usually want the best pose for HTVS speed
    command = (
        f"./gnina -r receptor_prepared.pdb -l {ligand_path} "
        f"--autobox_ligand reference_ligand.pdb --autobox_add 6 "
        f"--exhaustiveness 8 --cnn_scoring refined --seed 42 "
        f"-o {output_file} --num_modes 1"
    )
    subprocess.run(command, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ============================================================
# SECTION 6 — SCORE EXTRACTION & ANALYSIS
# ============================================================

In [ ]:
results = []
for result_file in glob.glob("docking_results/*_docked.sdf"):
    l_id = os.path.basename(result_file).replace("_docked.sdf", "")
    with open(result_file, "r") as f:
        text = f.read()
    
    vina = re.search(r'> <minimizedAffinity>\s*\n([-.\d]+)', text)
    cnn = re.search(r'> <CNNaffinity>\s*\n([-.\d]+)', text)
    
    if vina and cnn:
        results.append([l_id, float(vina.group(1)), float(cnn.group(1))])

results_df = pd.DataFrame(results, columns=["Ligand", "Vina_Affinity", "CNN_Affinity"])
results_df = results_df.sort_values(by="CNN_Affinity", ascending=False)
results_df.to_csv("KOR_HTVS_Results.csv", index=False)

print("\n--- TOP HITS ---")
print(results_df.head(10))

# ============================================================
# SECTION 7 — VISUALIZE TOP HIT
# ============================================================

In [ ]:
if not results_df.empty:
    top_hit = results_df.iloc[0]["Ligand"]
    view = py3Dmol.view(width=800, height=600)
    view.addModel(open("receptor_clean.pdb").read(), "pdb")
    view.setStyle({'cartoon': {'color': 'white', 'opacity': 0.8}})
    view.addModelsAsFrames(open(f"docking_results/{top_hit}_docked.sdf").read())
    view.setStyle({'model': 1}, {'stick': {'colorscheme': 'cyanCarbon'}})
    view.zoomTo()
    view.show()

# ============================================================
# SECTION 8 — DOWNLOAD RESULTS
# ============================================================

In [ ]:
!zip -r KOR_HTVS_Results.zip receptor_prepared.pdb docking_results KOR_HTVS_Results.csv
from google.colab import files
files.download("KOR_HTVS_Results.zip")